# FinSync · Notebook 01 — Gmail: Fetch & Explore

Step-by-step: connect to Gmail (**read-only**), pull candidate transaction emails
for **HBL / ABL / NayaPay**, and lay them out in a table so we can design the
parsers from your *real* emails.

**This notebook is read-only** — it never writes to the database. It's for
exploration so we can see the actual wording of your notification emails.

Run the cells top to bottom.

## Prerequisites (one-time)

1. **Install deps** into the backend venv:
   ```bash
   cd backend && source .venv/bin/activate
   pip install -r ../notebooks/requirements.txt
   ```
2. **Authorize Gmail** (read-only, once):
   ```bash
   python ../scripts/gmail_authorize.py
   ```
   This stores a refresh token at `backend/secrets/gmail_token.json`.
3. **Select the kernel**: top-right of this notebook → pick the interpreter at
   `backend/.venv`.

If Gmail isn't authorized yet, the connect cell below will tell you.

In [36]:
# Install notebook deps into THIS kernel (should be the backend/.venv interpreter).
# Using sys.executable guarantees pip targets the running kernel, not some other pip on PATH.
import sys
from pathlib import Path

_here = Path.cwd()
_req = next(
    (
        p / "requirements.txt"
        for p in (_here, _here / "notebooks", _here.parent / "notebooks")
        if (p / "requirements.txt").exists()
    ),
    None,
)
if _req is None:
    raise FileNotFoundError(f"Could not find notebooks/requirements.txt starting from {_here}")

!{sys.executable} -m pip install -q -r "{_req}"
print("Installed into:", sys.executable)

Installed into: /Users/danish/Python/FinSync/backend/.venv/bin/python


In [1]:
# Make the backend `app` package importable from this notebook.
import sys
from pathlib import Path


def _find_backend(start: Path) -> Path:
    # Walk up until we find a folder containing backend/app.
    for base in (start, *start.parents):
        if (base / "backend" / "app").is_dir():
            return base / "backend"
    raise FileNotFoundError(f"Could not find the 'backend' folder above {start}")


BACKEND = _find_backend(Path.cwd())
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import pandas as pd

pd.set_option("display.max_colwidth", 140)
print("backend on path:", BACKEND)
print("kernel python:  ", sys.executable)

backend on path: /Users/danish/Python/FinSync/backend
kernel python:   /Users/danish/Python/FinSync/backend/.venv/bin/python


In [2]:
# Build an authenticated, read-only Gmail service from the stored token.
from app.services.gmail.client import build_service

service = build_service()
if service is None:
    raise RuntimeError(
        "Gmail not authorized. Run this once from the backend venv:\n"
        "    python ../scripts/gmail_authorize.py"
    )

profile = service.users().getProfile(userId="me").execute()
print("Connected as:", profile.get("emailAddress"))
print("Total messages in mailbox:", profile.get("messagesTotal"))

Connected as: test.user@example.com
Total messages in mailbox: 13516


## 1. Choose what to search

Two ways to find your transaction emails:

- **By configured senders** — uses `email_senders` from `backend/providers.yaml`.
  (Right now those are `REPLACE_WITH_...` placeholders, so this returns nothing
  until you fill them in.)
- **Manual query** — set `MANUAL_QUERY` to any
  [Gmail search](https://support.google.com/mail/answer/7190) string to explore
  before you know the exact senders, e.g.
  `"from:hbl OR subject:(transaction OR transfer)"`.

In [39]:
# --- Parameters: edit these ---
PROVIDER = "abl"            # "hbl" | "abl" | "nayapay"
START_DATE = "2025/01/01"   # Gmail after:  YYYY/MM/DD  (or None)
END_DATE = None             # Gmail before: YYYY/MM/DD  (or None)
MAX_RESULTS = 100          # cap on messages to pull; raise/lower as needed
MANUAL_QUERY = "from:myabl@abl.com"   # search a specific sender; set to None to use providers.yaml

# Show the senders currently configured per provider.
from app.config import get_providers_config

providers = get_providers_config()
for key, cfg in providers.providers.items():
    print(f"{key:8s} email_senders={cfg.email_senders}")

hbl      email_senders=['replace_with_hbl_email_sender']
abl      email_senders=['replace_with_abl_email_sender']
nayapay  email_senders=['replace_with_nayapay_email_sender']


In [40]:
# Build the query and list matching message ids.
from app.services.gmail.client import build_query, list_message_ids

if MANUAL_QUERY:
    query = MANUAL_QUERY
else:
    cfg = providers.provider(PROVIDER)
    query = build_query(cfg.email_senders, after=START_DATE, before=END_DATE)
print("Gmail query:", query)

ids = list_message_ids(service, query, max_results=MAX_RESULTS)
print(f"matched {len(ids)} messages")
if not ids:
    print(
        "\nNo matches. Either fill real senders in backend/providers.yaml, "
        "or set MANUAL_QUERY above and re-run."
    )

Gmail query: from:myabl@abl.com
matched 100 messages


## 2. Fetch and tabulate

Pull each matched message and normalize it (sender, subject, date, snippet) into
a pandas table so we can scan them quickly.

In [41]:
# Fetch each message and build a DataFrame overview.
from app.services.gmail.client import fetch_email

emails = [fetch_email(service, mid) for mid in ids]

df = pd.DataFrame(
    [
        {
            "message_id": e.message_id,
            "date": e.received_at,
            "sender": e.sender,
            "subject": e.subject,
            "snippet": (e.body or "").strip().replace("\n", " ")[:160],
        }
        for e in emails
    ]
)
df

,message_id,date,sender,subject,snippet
0,1a0ee21e61f7292a,2026-09-29 22:06:32+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
1,1a0ee033ed6d4013,2026-09-29 21:33:02+05:00,myABL@abl.com,Allied Bank Transaction Alert,"<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4.01 Transitional//EN""><html><head><title>Allied Bank Limited</title><meta http-equiv=""Content-T..."
2,1a0ec1b4223623c0,2026-09-29 12:40:05+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
3,1a0eb4dc204b168c,2026-09-29 08:55:37+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
4,1a0e8feacb698818,2026-09-28 22:09:58+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
...,...,...,...,...,...
95,19faec01ceea9882,2026-07-29 21:40:54+05:00,myABL@abl.com,Allied Bank Transaction Alert,"<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4.01 Transitional//EN""><html><head><title>Allied Bank Limited</title><meta http-equiv=""Content-T..."
96,19fa7a93f8cdfd10,2026-07-28 12:38:36+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<html> <head> <title>Allied Bank Limited</title> <meta http-equiv=""Content-Type"" content=""text/html; charset=iso-8859-1"" /> <style type=..."
97,19fa7a8f8cb610e3,2026-07-28 12:38:16+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."
98,19fa7a8abe220603,2026-07-28 12:37:56+05:00,myABL@abl.com,myABL Fund Transfer Alert,"<!DOCTYPE html> <html> <head> <style> td { font-family: Tahoma, Verdana, Arial; font-size: 11px; font-weight: normal; color: #666666..."


## 3. Inspect one full email body

This is how we design the parsers: read the real structure of a transaction
email (amount, direction, reference, date format). Change `ROW` to look at a
different message from the table above.

In [42]:
ROW = 0  # index into the table above

if not emails:
    print("No emails fetched — adjust the search parameters above and re-run.")
else:
    e = emails[ROW]
    print("FROM:   ", e.sender)
    print("SUBJECT:", e.subject)
    print("DATE:   ", e.received_at)
    print("MSG_ID: ", e.message_id)
    print("-" * 70)
    print((e.body or "")[:3000])

FROM:    myABL@abl.com
SUBJECT: myABL Fund Transfer Alert
DATE:    2026-09-29 22:06:32+05:00
MSG_ID:  1a0ee21e61f7292a
----------------------------------------------------------------------
<!DOCTYPE html>
<html>
<head>
<style>
td { 
font-family: Tahoma, Verdana, Arial; 
font-size: 11px; 
font-weight: normal; 
color: #666666; 
text-decoration: none;
}

body {
font-family: Arial, sans-serif; /* Replace Arial with your desired font */
}
</style>
</head>
<body>
<table border="0" width="640" cellspacing="1" cellpadding="0" align="center" bgcolor="#999999">
<tbody>
<tr>
<td>
<table border="0" width="640" cellspacing="0" cellpadding="0" bgcolor="#ffffff">
<tbody>
<tr>
<td>&nbsp;</td>
</tr>
<tr>
<td>
<table border="0" width="600" cellspacing="0" cellpadding="0" align="center">
<tbody>
<tr>
<td>
<p><br /> Dear ACCOUNT HOLDER,<br /> <br /> PKR 17,000.00 has been received in your Account No: ***0017 on Tuesday  , 29-Sep-2026 at 10:06 PM.</p>
<p>Transaction details are as follow:</p>
<table style

## 4. Save an anonymized sample (optional but recommended)

Once you can see a real transaction email, save a **redacted** copy to
`notebooks/samples/` — remove account numbers, names, and any codes, but keep the
wording/structure. We'll use these to implement the parsers and their tests.

The cell below writes the currently-inspected email to a file. **Open it and
redact it** before sharing. (The `samples/` folder is gitignored.)

In [43]:
# Dump the currently-inspected email to samples/ for redaction.
# NOTE: this writes the RAW email — open the file and remove personal data.
SAMPLES = Path.cwd() / "samples"
SAMPLES.mkdir(exist_ok=True)

if emails:
    ROW = globals().get("ROW", 0)
    e = emails[ROW]
    out = SAMPLES / f"{PROVIDER}_email_{ROW}_RAW_REDACT_ME.txt"
    out.write_text(
        f"FROM: {e.sender}\nSUBJECT: {e.subject}\nDATE: {e.received_at}\n"
        f"{'-' * 70}\n{e.body or ''}",
        encoding="utf-8",
    )
    print("Wrote:", out)
    print("→ Open it and redact account numbers / names before sharing.")
else:
    print("Nothing to save yet.")

Wrote: /Users/danish/Python/FinSync/notebooks/samples/abl_email_0_RAW_REDACT_ME.txt
→ Open it and redact account numbers / names before sharing.


## Next steps

- Repeat sections 1–3 for `PROVIDER = "abl"` and `"nayapay"` to see each format.
- Fill the real senders/subject patterns into `backend/providers.yaml`.
- Save 1–2 redacted samples per provider into `notebooks/samples/`.

Then we'll add **Notebook 02 — Parse & Normalize**: run each provider parser on
these samples, build a normalized `Transaction` table, and preview
fingerprint/dedup — before wiring it to Postgres.

In [ ]:
# ABL
# Email: myabl@abl.com
# Card Transactions = Allied Bank Transaction Alert
# Funds Transfer = myABL Fund Transfer Alert

# Nayapay
# Email: service@nayapay.com
# Card Transactions = "Transaction" in subject
# Funds Transfer = "You Got" for Credit and "You Sent" for Debit


## 5. Verify the senders & subjects from the notes above

Before writing any parser, confirm the sender addresses and subject/body
patterns you jotted down actually match real messages in **your** mailbox.

The cell below encodes exactly what you wrote for **ABL** and **NayaPay**, then
runs each query against Gmail (read-only; it only lists ids + reads subjects —
no transaction bodies are pulled here). For each provider you should see a
non-zero count and sample subjects that match your description.


In [3]:
# Encode the notes above as Gmail queries, then verify each against the mailbox.
from app.services.gmail.client import list_message_ids, fetch_email

VERIFY_AFTER = "2025/01/01"  # Gmail after: YYYY/MM/DD (set None to search all time)

# sender + per-type query fragment, straight from your ABL / NayaPay notes.
PROVIDER_EMAILS = {
    "abl": {
        "sender": "myabl@abl.com",
        "types": {
            "card":     'subject:"Allied Bank Transaction Alert"',
            "transfer": 'subject:"myABL Fund Transfer Alert"',
        },
    },
    "nayapay": {
        "sender": "service@nayapay.com",
        "types": {
            "card":   "subject:(Transaction)",
            "credit": '"You Got"',   # funds received
            "debit":  '"You Sent"',  # funds sent
        },
    },
}


def _ids(q, cap=200):
    return list_message_ids(service, q, max_results=cap)


for prov, spec in PROVIDER_EMAILS.items():
    base = f"from:{spec['sender']}"
    if VERIFY_AFTER:
        base += f" after:{VERIFY_AFTER}"
    total = len(_ids(base))
    print(f"\n=== {prov.upper()}  {spec['sender']}  —  {total} msg since {VERIFY_AFTER} ===")
    for label, pat in spec["types"].items():
        q = f"{base} {pat}"
        mids = _ids(q)
        flag = "OK  " if mids else "NONE"
        print(f"  [{flag}] {label:9s} {len(mids):4d}   {q}")
        for mid in mids[:2]:
            print(f"           · {fetch_email(service, mid).subject}")



=== ABL  myabl@abl.com  —  200 msg since 2025/01/01 ===
  [OK  ] card       200   from:myabl@abl.com after:2025/01/01 subject:"Allied Bank Transaction Alert"
           · Allied Bank Transaction Alert
           · Allied Bank Transaction Alert
  [OK  ] transfer   200   from:myabl@abl.com after:2025/01/01 subject:"myABL Fund Transfer Alert"
           · myABL Fund Transfer Alert
           · myABL Fund Transfer Alert

=== NAYAPAY  service@nayapay.com  —  200 msg since 2025/01/01 ===
  [OK  ] card        75   from:service@nayapay.com after:2025/01/01 subject:(Transaction)
           · Card transaction of Rs. 3,142.96 failed
           · Card transaction of Rs. 3,142.96 failed
  [OK  ] credit     176   from:service@nayapay.com after:2025/01/01 "You Got"
           · You got Rs. 5,000 from Danish Zulfiqar 🎉
           · You got Rs. 5,000 from Danish Zulfiqar 🎉
  [OK  ] debit      200   from:service@nayapay.com after:2025/01/01 "You Sent"
           · You sent Rs. 1,076 to Muhammad Farooq 

## 6. Inspect one real body per transaction type

To design the parsers we need the exact wording of each email type. The cell
below pulls **one** example body for every (provider, type) pair found above and
prints it (truncated). Read these to see where the amount, date, reference,
counterparty, and running balance live.


In [4]:
import re


def _clean(body: str) -> str:
    # Collapse HTML to readable text so bank templates are legible.
    text = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", body or "", flags=re.S | re.I)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.I)
    text = re.sub(r"</(p|div|tr|table|h[1-6])>", "\n", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&nbsp;", " ", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    return text.strip()


for prov, spec in PROVIDER_EMAILS.items():
    base = f"from:{spec['sender']}"
    if VERIFY_AFTER:
        base += f" after:{VERIFY_AFTER}"
    for label, pat in spec["types"].items():
        mids = _ids(f"{base} {pat}")
        if not mids:
            continue
        em = fetch_email(service, mids[0])
        print("=" * 80)
        print(f"{prov.upper()} / {label}")
        print("SUBJECT:", em.subject)
        print("DATE:   ", em.received_at)
        print("-" * 80)
        print(_clean(em.body)[:1600])
        print()


ABL / card
SUBJECT: Allied Bank Transaction Alert
DATE:    2026-09-29 21:33:02+05:00
--------------------------------------------------------------------------------
Allied Bank Limited 

Transaction Alert 

Dear ACCOUNT HOLDER

A Debit transaction of USD 58.43 was made in your Account No: 0000****000017 (ACCOUNT HOLDER) in COMSATS, ISLAMABAD branch on Tuesday, 29 Sep 2026 at 9:27PM. 

Transaction description : PURCHASE GITHUB, INC. +18774484820 US
Instrument / Cheque# : 638702

If you need any further assistance, please contact our HelpLine on 042-111225225. 

Allied Bank

ABL / transfer
SUBJECT: myABL Fund Transfer Alert
DATE:    2026-10-06 23:38:59+05:00
--------------------------------------------------------------------------------
Dear ACCOUNT HOLDER,

PKR 1,000.00 have been sent from your Account No: ***0017 on Tuesday , 06-Oct-2026 at 11:38 PM through myABL.

Transaction details are as follow:

Transaction Description : 
RAAST Transfer 

Beneficiary Name : 
ALKHIDMAT FOUNDATION

In [5]:
# NayaPay bodies came back empty above — inspect the RAW payload to see what the
# message actually carries (plain vs html, length, and a readable sample).
import base64

spec = PROVIDER_EMAILS["nayapay"]
base = f"from:{spec['sender']}"
if VERIFY_AFTER:
    base += f" after:{VERIFY_AFTER}"


def _walk_parts(payload, depth=0):
    mime = payload.get("mimeType", "")
    size = payload.get("body", {}).get("size", 0)
    has_data = bool(payload.get("body", {}).get("data"))
    print("  " * depth + f"- {mime}  size={size}  data={has_data}")
    for p in payload.get("parts", []) or []:
        _walk_parts(p, depth + 1)


def _raw_full(service, mid):
    msg = service.users().messages().get(userId="me", id=mid, format="full").execute()
    return msg


for label in ("card", "credit", "debit"):
    mids = _ids(f"{base} {spec['types'][label]}")
    if not mids:
        continue
    msg = _raw_full(service, mids[0])
    print("=" * 80)
    print(f"NAYAPAY / {label}")
    print("SUBJECT:", next((h["value"] for h in msg["payload"]["headers"] if h["name"].lower() == "subject"), ""))
    print("snippet:", msg.get("snippet", "")[:300])
    print("mime tree:")
    _walk_parts(msg["payload"])
    print("cleaned text/html sample:")
    print(_clean(fetch_email(service, mids[0]).body)[:1200] or "    <empty>")
    print()


NAYAPAY / card
SUBJECT: Card transaction of Rs. 3,142.96 failed
snippet: Hi Danish! An attempted transaction of Rs. 3142.96 at NEXSYS AI LLC London GB on your Visa card ****1234 has been declined due to regulatory compliance. Please reach out to our Customer Support at 111-
mime tree:
- multipart/mixed  size=0  data=False
  - multipart/related  size=0  data=False
    - multipart/alternative  size=0  data=False
      - text/plain  size=245  data=True
      - text/html  size=13362  data=True
cleaned text/html sample:
An attempted transaction of Rs. 3,142.96 at NEXSYS AI LLC London GB on your Visa card ****1234 has been declined due to regulatory compliance. Please reach out to our Customer Support at 111-222-729 or through in-app chat for further assistance.

NAYAPAY / credit
SUBJECT: You got Rs. 5,000 from Danish Zulfiqar 🎉
snippet: Cha-Ching! Danish Zulfiqar HBL-5003 09 Oct 2026, 07:29 PM Rs. 5000 AMOUNT DETAILS Amount Received Rs. 5000 Service Fee (Incl. Tax) Rs. 0 Total Amount Rs. 50

In [6]:
# Decode the text/html part directly (the client picks the near-empty text/plain
# part for NayaPay) and show the readable text so we can design the parser.
def _find_html(payload):
    if payload.get("mimeType") == "text/html" and payload.get("body", {}).get("data"):
        return payload["body"]["data"]
    for p in payload.get("parts", []) or []:
        found = _find_html(p)
        if found:
            return found
    return None


for label in ("card", "credit", "debit"):
    mids = _ids(f"{base} {spec['types'][label]}")
    if not mids:
        continue
    msg = _raw_full(service, mids[0])
    data = _find_html(msg["payload"])
    html = base64.urlsafe_b64decode(data.encode()).decode("utf-8", "replace") if data else ""
    print("=" * 80)
    print(f"NAYAPAY / {label}")
    print(_clean(html)[:1600] or "<empty>")
    print()


NAYAPAY / card
NayaPay Email 

 --> 

 Hi Danish! 

 An attempted transaction of Rs. 3,142.96 at NEXSYS AI LLC London GB on your Visa card ****1234 has been declined due to regulatory compliance. Please reach out to our Customer Support at 111-222-729 or through in-app chat for further assistance. 

 " " " " 

 Do not reply to this email. Responses to this email are not monitored. 
 NayaPay (Private) Limited is authorised and regulated by the State Bank of Pakistan as an Electronic Money Institution.
 (021) 111-222-729 | www.nayapay.com | support@nayapay.com 
 NayaPay, its employees or agents will never ask you for your password, MPIN, OTP or debit card PIN, via call, email, SMS, web links or social media. For security purposes, please do not share this sensitive information with anyone.

NAYAPAY / credit
NayaPay Email 

 Cha-Ching! 

 --> 

 Danish Zulfiqar 

 HBL-5003 

 09 Oct 2026, 07:29 PM 

 Rs. 5,000 

 AMOUNT DETAILS 

 Amount Received 

 Rs. 5,000 

 Service Fee (Incl. Tax) 



## 7. HBL — fetch & sample

HBL transaction alerts come from **`digital.banking@hbl.com`**. Confirmed subjects:

- `HBL Mobile | Transfer to Other Bank` — outgoing funds transfer (debit)
- `HBL Mobile | Mobile Bundle` — mobile bundle / top-up purchase (debit)

Incoming funds and debit-card purchases arrive by **SMS** (handled separately
later), so we only wire the two outgoing-email types here.

The cells below reuse the helpers defined above (`_ids`, `fetch_email`,
`_clean`, `_raw_full`, `_find_html`, `VERIFY_AFTER`, `service`), so run sections
1–6 first (at least the Gmail-service and helper cells).


In [8]:
# HBL alerts come from digital.banking@hbl.com. Gmail's subject: search mangles
# the "|" in HBL subjects, so instead pull the sender's mail and tally the exact
# subjects client-side to see what's really there.
from collections import Counter

HBL_SENDER = "digital.banking@hbl.com"
hbl_base = f"from:{HBL_SENDER}"
if VERIFY_AFTER:
    hbl_base += f" after:{VERIFY_AFTER}"

hbl_ids = _ids(hbl_base, cap=200)
hbl_emails = [fetch_email(service, mid) for mid in hbl_ids]
subjects = Counter((e.subject or "").strip() for e in hbl_emails)

print(f"=== HBL  {HBL_SENDER}  —  {len(hbl_ids)} msg since {VERIFY_AFTER} ===")
for subj, n in subjects.most_common():
    print(f"  {n:4d}  {subj!r}")


Gmail API 403, backing off 1.0s (attempt 1/6)
Gmail API 403, backing off 2.0s (attempt 2/6)
Gmail API 403, backing off 4.0s (attempt 3/6)


=== HBL  digital.banking@hbl.com  —  200 msg since 2025/01/01 ===
    81  'HBL Mobile | Login'
    51  'HBL Mobile | Transfer to Other Bank'
    25  'HBL Digital | Login'
    17  'HBL Digital | Send Money to Other Bank via Raast'
    12  'HBL Mobile | Mobile Bundle'
     5  'HBL Digital | Bill Payment'
     3  'HBL Mobile | P2M Payment'
     2  'Your Account Statement for 08747*******03'
     1  'HBL Mobile | Prepaid Top-up'
     1  'HBL Digital | P2M Payment'
     1  'HBL Digital | One Time Password (OTP) for Merchant Payment'
     1  'HBL Digital | Failed Login Notification on HBL MB/IB'


In [9]:
# Print one real body per target HBL subject and save a RAW sample for redaction.
# Reuses `hbl_emails` already fetched above (no extra list calls). HBL templates
# are HTML; if the text/plain part is empty, decode the text/html part directly.
import base64

# subject -> short label used for the sample filename.
HBL_TARGETS = {
    "HBL Mobile | Transfer to Other Bank": "transfer",
    "HBL Mobile | Mobile Bundle": "bundle",
    "HBL Digital | Send Money to Other Bank via Raast": "raast",
}


def _hbl_body(em) -> str:
    cleaned = _clean(em.body or "")
    if cleaned.strip():
        return cleaned
    data = _find_html(_raw_full(service, em.message_id)["payload"])
    html = base64.urlsafe_b64decode(data.encode()).decode("utf-8", "replace") if data else ""
    return _clean(html)


SAMPLES = Path.cwd() / "samples"
SAMPLES.mkdir(exist_ok=True)

by_subject = {}
for e in hbl_emails:
    by_subject.setdefault((e.subject or "").strip(), e)

for subject, label in HBL_TARGETS.items():
    em = by_subject.get(subject)
    if em is None:
        print(f"HBL / {label}: no messages for {subject!r}\n")
        continue
    body = _hbl_body(em)
    print("=" * 80)
    print(f"HBL / {label}")
    print("SUBJECT:", em.subject)
    print("DATE:   ", em.received_at)
    print("-" * 80)
    print(body[:1800])
    out = SAMPLES / f"hbl_email_{label}_RAW_REDACT_ME.txt"
    out.write_text(
        f"FROM: {em.sender}\nSUBJECT: {em.subject}\nDATE: {em.received_at}\n"
        f"{'-' * 70}\n{body}",
        encoding="utf-8",
    )
    print(f"\n→ wrote {out.name} — redact account numbers / names before sharing.\n")


HBL / transfer
SUBJECT: HBL Mobile | Transfer to Other Bank
DATE:    2026-10-08 04:02:11+00:00
--------------------------------------------------------------------------------
Dear Customer, 

Your funds transfer was successful. Transaction details are as follows: 

Amount 
PKR 5,000 

Receiver Account Title 
Danish Zulfiqar 

Receiver Bank Name 
NayaPay 

From Account Title 
Danish Zulfiqar 

From Acc.No/IBAN 
*5003 

From Bank Name 
HBL / Konnect 

Date and Time 
08-Oct-2026 09:02 

Channel 
via Raast 

Transaction ID 
1227054068 

If you have not performed this transaction, please call HBL PhoneBanking at 021-111-111-425.

Important Notice: HBL will never ask for your password or any security information through email, SMS, phone etc.

Disclaimer: This email contains privileged and/or confidential information intended only for the use of the addressee. If you are not the intended recipient of this message, you are strictly prohibited to read, copy, distribute, discuss, disseminate
i